# Colab Setup with Google Drive

**建议方案**: 使用 Google Drive 持久化存储数据和模型

优点：
- ✅ Session 断开后数据不丢失
- ✅ 多个 notebook 间共享数据
- ✅ kaggle.json 存一次，永久使用
- ✅ 节省 Colab 临时磁盘空间（只有 ~200GB，之后自动删除）

**注意**: Google Drive 访问较慢，适合存储，不适合频繁I/O的大数据

## Step 1: 挂载 Google Drive

In [ ]:
from google.colab import drive
import os

# 挂载 Google Drive（会弹出授权窗口）
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive/MyDrive")

## Step 2: 准备 Google Drive 目录结构

In [ ]:
from pathlib import Path

# 创建项目目录结构
GDRIVE_BASE = Path("/content/drive/MyDrive/3drna_cc")
GDRIVE_BASE.mkdir(exist_ok=True)

dirs = [
    GDRIVE_BASE / "data",
    GDRIVE_BASE / "models",
    GDRIVE_BASE / "output",
    GDRIVE_BASE / "credentials",
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ {d.name}")

print(f"\n项目目录: {GDRIVE_BASE}")

## Step 3: 上传 kaggle.json 到 Google Drive（一次性）

In [ ]:
from google.colab import files
import shutil

cred_dir = Path("/content/drive/MyDrive/3drna_cc/credentials")
kaggle_json = cred_dir / "kaggle.json"

if kaggle_json.exists():
    print("✓ kaggle.json already in Google Drive")
else:
    print("请从 https://www.kaggle.com/account/login")
    print("  → Account → Create New API Token")
    print("  下载 kaggle.json，然后上传:")
    uploaded = files.upload()
    
    if "kaggle.json" in uploaded:
        shutil.move("kaggle.json", str(kaggle_json))
        print(f"✓ 已保存到 {kaggle_json}")
    else:
        print("✗ 未找到 kaggle.json")

## Step 4: 配置 Kaggle API（每个 session 执行一次）

In [ ]:
import os
from pathlib import Path

# 从 Google Drive 复制 kaggle.json 到 Colab 默认位置
gdrive_kaggle = Path("/content/drive/MyDrive/3drna_cc/credentials/kaggle.json")
colab_kaggle = Path("/root/.kaggle/kaggle.json")

if gdrive_kaggle.exists():
    Path("/root/.kaggle").mkdir(exist_ok=True)
    import shutil
    shutil.copy(str(gdrive_kaggle), str(colab_kaggle))
    os.chmod(str(colab_kaggle), 0o600)
    print("✓ kaggle.json 已配置")
    # 验证
    !kaggle competitions list | head -3
else:
    print("✗ 未找到 kaggle.json，请先上传到 Google Drive")

## Step 5: 克隆代码库到 Colab 临时存储

In [ ]:
import subprocess
import os

# 代码库放在 Colab 临时存储（快速）
REPO_DIR = "/content/3drna_cc"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/scyoyo/3drna_cc.git {REPO_DIR}
    print(f"✓ 代码库克隆到 {REPO_DIR}")
else:
    print(f"✓ 代码库已存在于 {REPO_DIR}")

%cd {REPO_DIR}

## Step 6: 安装依赖

In [ ]:
!pip install -q -r requirements.txt
print("✓ 依赖安装完成")

# 验证关键包
import protenix
import torch
print(f"✓ Protenix {protenix.__version__ if hasattr(protenix, '__version__') else 'installed'}")
print(f"✓ PyTorch {torch.__version__} (CUDA: {torch.cuda.is_available()})")

## Step 7: 验证配置（关键！）

In [ ]:
from src.config import IN_COLAB, GDRIVE_AVAILABLE, DATA_DIR, OUTPUT_DIR, MODEL_DIR

print(f"环境检测:")
print(f"  IN_COLAB: {IN_COLAB}")
print(f"  GDRIVE_AVAILABLE: {GDRIVE_AVAILABLE}")
print()
print(f"数据路径配置:")
print(f"  DATA_DIR: {DATA_DIR}")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")
print(f"  MODEL_DIR: {MODEL_DIR}")
print()
print(f"Google Drive 可用空间: {os.popen('df -h /content/drive | tail -1 | awk "{print $4}"').read().strip()}")

## Step 8: 下载竞赛数据到 Google Drive（第一次）

In [ ]:
import os
from src.config import DATA_DIR

# 检查数据是否已存在
data_files = [
    DATA_DIR / "train_sequences.csv",
    DATA_DIR / "train_labels.csv",
    DATA_DIR / "validation_sequences.csv",
]

if all(f.exists() for f in data_files):
    print("✓ 竞赛数据已在 Google Drive")
else:
    print("下载竞赛数据到 Google Drive...")
    print("💾 首次下载 ~50GB (取决于网络)")
    
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    # 仅下载必要文件（不下载 310GB 的 PDB_RNA）
    !kaggle competitions download -c stanford-rna-3d-folding-2 \
      -p {DATA_DIR} \
      -f train_sequences.csv \
      -f train_labels.csv \
      -f validation_sequences.csv \
      -f validation_labels.csv \
      -f test_sequences.csv \
      -f sample_submission.csv
    
    # 解压
    for zf in DATA_DIR.glob("*.zip"):
        !unzip -q {zf} -d {DATA_DIR}
        zf.unlink()
    
    print("✓ 数据下载完成！")
    print(f"\n下载的文件:")
    !ls -lh {DATA_DIR}/*.csv 2>/dev/null | awk '{print $9, $5}'

## Step 9: 下载 MSA 文件（可选，但推荐）

In [ ]:
msa_dir = DATA_DIR / "MSA"

if (msa_dir / "README.txt").exists() or len(list(msa_dir.glob("*.fasta"))) > 0:
    print(f"✓ MSA 文件已在 {msa_dir}")
else:
    print("下载 MSA 文件...")
    print("💾 MSA 大小 ~10GB")
    
    !kaggle competitions download -c stanford-rna-3d-folding-2 \
      -p {DATA_DIR} \
      -f MSA
    
    # 解压
    !unzip -q {DATA_DIR}/MSA.zip -d {DATA_DIR}
    !rm -f {DATA_DIR}/MSA.zip
    
    print("✓ MSA 下载完成！")
    print(f"MSA 文件数: {len(list(msa_dir.glob('*.fasta')))}")

## Step 10: 完整性检查

In [ ]:
from pathlib import Path
import pandas as pd

print("="*60)
print("✓ Colab + Google Drive 设置完成！")
print("="*60)
print()
print("Google Drive 目录结构:")
!tree -L 2 /content/drive/MyDrive/3drna_cc 2>/dev/null || find /content/drive/MyDrive/3drna_cc -maxdepth 2 -type d | sort
print()

# 数据统计
train_seq = pd.read_csv(DATA_DIR / "train_sequences.csv")
train_labels = pd.read_csv(DATA_DIR / "train_labels.csv")
msa_dir = DATA_DIR / "MSA"
msa_count = len(list(msa_dir.glob("*.fasta"))) if msa_dir.exists() else 0

print(f"数据统计:")
print(f"  Train 序列: {len(train_seq)}")
print(f"  Train 标签行: {len(train_labels)}")
print(f"  MSA 文件: {msa_count}")
print()
print(f"磁盘占用: {os.popen(f'du -sh {GDRIVE_BASE}').read().strip()}")
print()
print(f"下一步: 打开 01_setup_and_explore.ipynb 继续!")